# 1. Relationship Analysis

The univariate analysis provided a detailed understanding of each feature individually, including missingness, distribution shape, unusual values, category structure, and preliminary handling considerations.

The next stage examines how important applicant, financial, credit-history, household, and application characteristics behave together.

Each section will begin with a practical analytical question, use appropriate statistical and visual evidence, and conclude with a clear interpretation and next-step implication.

The objectives of this notebook are to:

- identify meaningful relationships among related features;
- understand how applicant characteristics combine to influence credit risk;
- distinguish complementary information from redundant information;
- identify useful ratios, interactions, segments, and special-value treatments;
- convert the findings into an implementation-ready feature engineering plan.

No feature transformation or removal will be applied during this analysis.

## 1.1 Relationship Analysis Setup

This section loads the integrated datasets and the consolidated univariate evidence required for the relationship analysis.

The univariate master file will be used as the central feature reference so that earlier reports do not need to be reviewed repeatedly.

### Imports and configuration

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from scipy import stats
ID_COL, TARGET_COL = "SK_ID_CURR", "TARGET"
RANDOM_STATE, PLOT_SAMPLE_SIZE = 42, 30_000

DATA_DIR = Path("../Data/Processed")
TRAIN_PATH = DATA_DIR / "home_credit_train_integrated.parquet"
TEST_PATH = DATA_DIR / "home_credit_test_integrated.parquet"
UNIVARIATE_MASTER_PATH = Path("Eda_report/Final_Univariate/master_file_univariate.csv")

RELATIONSHIP_REPORT_DIR = Path("Relationship_report")
RELATIONSHIP_PLOT_DIR = Path("Relationship_plots")
RELATIONSHIP_REPORT_DIR.mkdir(parents=True, exist_ok=True)
RELATIONSHIP_PLOT_DIR.mkdir(parents=True, exist_ok=True)

### Load data and feature groups

In [16]:
train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)
univariate_master = pd.read_csv(UNIVARIATE_MASTER_PATH, low_memory=False).drop_duplicates("feature")

candidate_features = [feature for feature in univariate_master["feature"] if feature in train_df.columns and feature not in [ID_COL, TARGET_COL]]

NUMERIC_CATEGORIES = {
    "continuous_amount_numeric", "continuous_stat_numeric", "score_index_numeric", "property_normalized_numeric",
    "discrete_count_numeric", "ratio_rate_numeric", "date_duration_numeric", "overdue_delay_numeric", "cyclical_time"
}
BINARY_CATEGORIES = {"binary_flag", "categorical_binary"}

numeric_features = univariate_master.loc[univariate_master["master_feature_category"].isin(NUMERIC_CATEGORIES) & univariate_master["feature"].isin(candidate_features), "feature"].tolist()
binary_features = univariate_master.loc[univariate_master["master_feature_category"].isin(BINARY_CATEGORIES) & univariate_master["feature"].isin(candidate_features), "feature"].tolist()
nominal_features = univariate_master.loc[univariate_master["master_feature_category"].eq("categorical_nominal") & univariate_master["feature"].isin(candidate_features), "feature"].tolist()
ordinal_features = univariate_master.loc[univariate_master["master_feature_category"].eq("categorical_ordinal") & univariate_master["feature"].isin(candidate_features), "feature"].tolist()

relationship_setup_summary = pd.DataFrame({
    "Item": ["Train rows", "Test rows", "Candidate features", "Numeric features", "Binary features", "Nominal features", "Ordinal features", "Baseline default rate"],
    "Value": [len(train_df), len(test_df), len(candidate_features), len(numeric_features), len(binary_features), len(nominal_features), len(ordinal_features), f"{train_df[TARGET_COL].mean() * 100:.2f}%"]
})

relationship_setup_summary

,Item,Value
0,Train rows,307511
1,Test rows,48744
2,Candidate features,2566
3,Numeric features,2379
4,Binary features,174
5,Nominal features,10
6,Ordinal features,3
7,Baseline default rate,8.07%


## 1.2 Is the Requested Credit Affordable Relative to Applicant Income?

The univariate analysis described income, credit amount, annuity, and goods price separately.

However, the absolute loan amount alone does not indicate whether the requested credit is affordable. The same credit amount may represent a manageable obligation for one applicant and a severe financial burden for another.

### Research question

Does default risk increase when the requested credit is large relative to the applicant's income?

### Variables considered

- Applicant capacity: `AMT_INCOME_TOTAL`
- Credit exposure: `AMT_CREDIT`
- Periodic repayment pressure: `AMT_ANNUITY`
- Financed asset value: `AMT_GOODS_PRICE`
- Loan structure: `NAME_CONTRACT_TYPE`
- Income source: `NAME_INCOME_TYPE`
- Credit outcome: `TARGET`

### Analytical approach

We will compare applicants across income and credit-burden segments, measure default-rate changes, and examine whether the relationship remains consistent across contract types.

The analysis uses relative financial-burden measures rather than relying only on absolute monetary values.

In [18]:
affordability_cols = ["TARGET", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "NAME_CONTRACT_TYPE", "NAME_INCOME_TYPE"]
affordability_df = train_df[affordability_cols].replace([np.inf, -np.inf], np.nan).dropna(subset=["TARGET", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]).copy()
affordability_df = affordability_df.loc[(affordability_df["AMT_INCOME_TOTAL"] > 0) & (affordability_df["AMT_CREDIT"] > 0) & (affordability_df["AMT_ANNUITY"] > 0)].copy()

affordability_df["credit_to_income_ratio"] = affordability_df["AMT_CREDIT"] / affordability_df["AMT_INCOME_TOTAL"]
affordability_df["annuity_to_income_ratio"] = affordability_df["AMT_ANNUITY"] / affordability_df["AMT_INCOME_TOTAL"]
affordability_df["goods_to_income_ratio"] = affordability_df["AMT_GOODS_PRICE"] / affordability_df["AMT_INCOME_TOTAL"]

for col in ["AMT_INCOME_TOTAL", "credit_to_income_ratio", "annuity_to_income_ratio"]:
    lower, upper = affordability_df[col].quantile([0.01, 0.99])
    affordability_df = affordability_df.loc[affordability_df[col].between(lower, upper)].copy()

affordability_df["income_bin"] = pd.qcut(affordability_df["AMT_INCOME_TOTAL"], q=8, labels=False, duplicates="drop") + 1
affordability_df["burden_bin"] = pd.qcut(affordability_df["credit_to_income_ratio"], q=8, labels=False, duplicates="drop") + 1

affordability_surface = affordability_df.groupby(["income_bin", "burden_bin"], observed=True).agg(default_rate=("TARGET", "mean"), applicant_count=("TARGET", "size"), median_income=("AMT_INCOME_TOTAL", "median"), median_burden=("credit_to_income_ratio", "median")).reset_index()
affordability_surface["default_rate_pct"] = affordability_surface["default_rate"] * 100
risk_matrix = affordability_surface.pivot(index="income_bin", columns="burden_bin", values="default_rate_pct")
support_matrix = affordability_surface.pivot(index="income_bin", columns="burden_bin", values="applicant_count")

fig = px.imshow(risk_matrix, text_auto=".1f", aspect="auto", labels={"x": "Credit-to-Income Burden Bin", "y": "Income Bin", "color": "Default Rate (%)"}, title="Default Risk Across Income and Credit-Burden Segments")
fig.update_layout(height=620)
fig.write_html(RELATIONSHIP_PLOT_DIR / "1.2_income_credit_burden_risk_surface.html")
fig.show()

contract_risk = affordability_df.groupby(["NAME_CONTRACT_TYPE", "burden_bin"], observed=True).agg(default_rate=("TARGET", "mean"), applicant_count=("TARGET", "size"), median_burden=("credit_to_income_ratio", "median")).reset_index()
contract_risk["default_rate_pct"] = contract_risk["default_rate"] * 100

fig = px.line(contract_risk, x="burden_bin", y="default_rate_pct", color="NAME_CONTRACT_TYPE", markers=True, hover_data=["applicant_count", "median_burden"], labels={"burden_bin": "Credit-to-Income Burden Bin", "default_rate_pct": "Default Rate (%)", "NAME_CONTRACT_TYPE": "Contract Type"}, title="Credit-Burden Risk Trend by Contract Type")
fig.update_layout(height=520)
fig.write_html(RELATIONSHIP_PLOT_DIR / "1.2_contract_type_burden_risk_trend.html")
fig.show()

overall_burden_risk = affordability_df.groupby("burden_bin", observed=True).agg(default_rate=("TARGET", "mean"), applicant_count=("TARGET", "size")).reset_index()
lowest_rate, highest_rate = overall_burden_risk.iloc[0]["default_rate"], overall_burden_risk.iloc[-1]["default_rate"]
trend_corr = stats.spearmanr(overall_burden_risk["burden_bin"], overall_burden_risk["default_rate"]).statistic
highest_risk_segment = affordability_surface.loc[affordability_surface["default_rate"].idxmax()]

affordability_answer = pd.DataFrame({
    "Measure": ["Baseline default rate", "Lowest burden-bin default rate", "Highest burden-bin default rate", "High-vs-low burden lift", "Burden-risk Spearman correlation", "Highest-risk income bin", "Highest-risk burden bin", "Highest-risk segment default rate", "Highest-risk segment applicants"],
    "Value": [f"{train_df[TARGET_COL].mean() * 100:.2f}%", f"{lowest_rate * 100:.2f}%", f"{highest_rate * 100:.2f}%", f"{highest_rate / lowest_rate:.2f}x", f"{trend_corr:.3f}", int(highest_risk_segment["income_bin"]), int(highest_risk_segment["burden_bin"]), f"{highest_risk_segment['default_rate_pct']:.2f}%", int(highest_risk_segment["applicant_count"])]
})

affordability_surface.to_csv(RELATIONSHIP_REPORT_DIR / "1.2_income_credit_burden_relationship.csv", index=False, encoding="utf-8-sig")
affordability_answer

,Measure,Value
0,Baseline default rate,8.07%
1,Lowest burden-bin default rate,7.28%
2,Highest burden-bin default rate,7.03%
3,High-vs-low burden lift,0.97x
4,Burden-risk Spearman correlation,-0.190
5,Highest-risk income bin,4
6,Highest-risk burden bin,3
7,Highest-risk segment default rate,13.29%
8,Highest-risk segment applicants,474


### Affordability Relationship — Findings, Concerns, and Current Decision

We began with a simple question:

> Is a large loan risky by itself, or does risk depend on how large the loan is relative to the applicant's income?

To answer this, we compared applicant income, requested credit, contract type, and observed default rate.

The main analytical measure was the **credit-to-income ratio**:

**Credit-to-Income Ratio = Requested Credit / Applicant Income**

A higher ratio means that the requested credit is larger relative to the applicant's reported income.

For example, a loan of 500,000 may be manageable for an applicant earning 500,000, but much more demanding for someone earning 100,000.

---

#### What Did We Find?

Hmm... the first important result is that affordability risk is not explained by loan amount alone.

The cash-loan pattern shows that default risk initially increases as the credit-to-income burden rises. The observed default rate grows from approximately 7.5% in the lowest burden group to around 9.6% in the middle burden groups.

However, the risk does not continue rising in the highest burden groups. Instead, it begins to decline.

This means that the relationship is **non-linear**:

**Higher Burden ≠ Continuously Higher Risk**

The heatmap confirms the same pattern from another perspective.

The highest default rates are not concentrated only among the lowest-income applicants or only among applicants with the largest relative burden. Instead, several middle-income and moderate-to-high burden combinations show elevated risk.

Notable observed segments include:

- Income Bin 2 and Burden Bin 5: approximately 11.6% default;
- Income Bin 3 and Burden Bin 5: approximately 10.8%;
- Income Bin 5 and Burden Bins 4–5: approximately 10.5%–10.6%;
- Income Bin 4 and Burden Bin 3: approximately 13.3%, the highest visible cell.

Yeah... this suggests that risk emerges from the **combination of income and credit burden**, rather than from either variable independently.

---

#### What Additional Pattern Did We Observe?

The contract-type comparison reveals that cash and revolving loans behave differently.

Cash loans maintain a higher default rate across all burden groups.

Revolving loans show a declining default rate as the calculated burden increases. At first glance, this could appear to suggest that higher revolving-credit burden is safer.

Uhh... that interpretation would be too simplistic.

For revolving products, `AMT_CREDIT` may represent an available credit limit rather than the amount actually used. A customer may have a large credit limit while carrying only a small outstanding balance.

Therefore, the same credit-to-income ratio may not represent the same financial pressure for cash and revolving loans.

The analysis indicates that:

- cash-loan affordability can be interpreted more directly;
- revolving-loan affordability requires utilization or balance information;
- one universal burden rule should not be applied to both products.

---

#### What Problems or Limitations Did We Identify?

Several issues prevent us from converting these results into a final feature-engineering rule immediately.

**1. The relationship is not monotonic**

Risk rises, peaks, and then declines. Therefore, a simple linear interpretation would be misleading.

**2. Extreme burden does not always show extreme risk**

The highest burden bins often show lower default rates than the middle bins. This may reflect product structure, applicant selection, stronger financial profiles, or unequal segment support.

**3. Cash and revolving loans have different business meanings**

The same `AMT_CREDIT` value may represent an actual financed amount for cash loans but an available credit limit for revolving loans.

**4. Some high-risk cells may have limited support**

A high default percentage is not automatically reliable.

For example:

- 2 defaults among 10 applicants produce a 20% default rate;
- 1,000 defaults among 10,000 applicants produce a 10% default rate.

The first rate is higher, but it is based on much less evidence.

**5. Income and credit do not explain the full applicant profile**

External scores, employment stability, previous credit history, family burden, and asset ownership may change the observed affordability pattern.

**6. Quantile bins are relative groups**

Burden Bin 8 does not mean that the ratio equals eight. It represents the highest relative burden group within this dataset.

---

#### What Does the Evidence Mean?

The current evidence supports the following interpretation:

**Default Risk = f(Income, Credit Burden, Contract Type, Applicant Strength)**

In practical terms:

- a large loan is not automatically risky;
- a low-income applicant is not automatically risky;
- risk rises when the loan structure becomes difficult relative to the applicant's financial capacity;
- the same burden can have different meanings across contract types and applicant groups;
- middle-income applicants with moderate-to-high burden may contain important risk segments;
- high-income applicants may remain relatively safe even with large requested credit, depending on their broader financial profile.

---

#### Current Feature-Level Decision

| Feature or relationship | Current decision |
|---|---|
| `AMT_INCOME_TOTAL` | Retain as a core applicant-capacity variable |
| `AMT_CREDIT` | Retain as a core exposure variable |
| `NAME_CONTRACT_TYPE` | Retain as an important relationship modifier |
| Credit-to-income ratio | Carry forward as a feature-engineering candidate |
| Income × burden interaction | Carry forward for multivariate validation |
| Contract type × burden interaction | Analyze separately for cash and revolving loans |
| Universal burden threshold | Do not define yet |
| Immediate transformation | Not approved yet |
| Immediate feature removal | Not supported |

---

#### What Should We Check Next?

The affordability relationship should now be examined with additional applicant-strength variables.

The next useful questions are:

- Does a strong external score reduce the risk associated with high burden?
- Does short employment history make the same burden more dangerous?
- Does absence of bureau history change the meaning of the ratio?
- Does family size reduce repayment capacity at similar income levels?
- Is annuity-to-income more informative than credit-to-income?
- Does revolving-credit utilization explain the unusual declining pattern?

---

#### Final Answer for This Stage

Hmm... we found meaningful affordability information, but not a simple rule.

The credit-to-income ratio appears useful, and the interaction between income, burden, and contract type deserves further analysis.

However, the relationship is non-linear, product-dependent, and potentially affected by subgroup support and other applicant characteristics.

Therefore, the current conclusion is:

> Keep the original financial variables, retain the credit-to-income ratio as a candidate, and continue multivariate validation before defining any final feature-engineering treatment.